# Análisis Exploratorio de Tickets de Soporte Técnico

## Proyecto de Pandas & Data Visualization

Este notebook presenta un análisis exploratorio de datos sobre tickets de soporte al cliente. El objetivo es analizar patrones operativos relacionados con tipos de tickets, prioridad, canales de atención, tiempos de resolución, productos con mayor carga de soporte y satisfacción del cliente.

El análisis sigue un flujo reproducible:

1. Carga del dataset original.
2. Revisión inicial de estructura y calidad de datos.
3. Limpieza y transformación de columnas.
4. Creación de nuevas variables analíticas.
5. Visualizaciones con intención.
6. Hallazgos principales y conclusiones.

Este proyecto utiliza un enfoque modular, separando parte de la lógica en archivos dentro de `src/`, para que el análisis pueda ejecutarse tanto desde este notebook como desde `main.py`.

## 1. Objetivo del análisis

El objetivo de este proyecto es analizar un dataset de tickets de soporte técnico para responder preguntas como:

- ¿Qué tipos de tickets son más frecuentes?
- ¿Qué prioridades concentran mayor volumen de tickets?
- ¿Los tickets críticos tardan más o menos en resolverse?
- ¿Qué canales de atención tienen mejor satisfacción promedio?
- ¿Existe relación entre tiempo de resolución y satisfacción del cliente?
- ¿Qué productos generan mayor carga de soporte?

Este tipo de análisis es útil para equipos de soporte, Service Desk, operaciones IT y gestión de calidad, ya que permite identificar oportunidades de mejora en procesos, priorización y experiencia del usuario.

In [2]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Asegurar que el notebook pueda importar módulos desde src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import RAW_PATH, OUT_PATH, FIGURES_PATH
from src.io import load_csv
from src.cleaning import clean
from src.features import build_features
from src.utils import validate_raw_dataset, validate_clean_dataset
from src.viz import (
    plot_tickets_by_type,
    plot_tickets_by_priority,
    plot_resolution_time_by_priority,
    plot_satisfaction_by_channel,
    plot_resolution_vs_satisfaction,
    plot_top_products_by_tickets,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

sns.set_theme(style="whitegrid", context="notebook")

print("Setup completo. Listo para el análisis.")

Setup completo. Listo para el análisis.


## 2. Dataset utilizado

El dataset utilizado es **Customer Support Ticket Dataset**, descargado desde Kaggle:

https://www.kaggle.com/datasets/suraj520/customer-support-ticket-dataset

El archivo CSV fue guardado dentro de:

`data/raw/customer_support_tickets.csv`

El dataset contiene tickets de soporte al cliente con información como producto comprado, tipo de ticket, prioridad, canal de atención, estado del ticket, fechas de respuesta/resolución y satisfacción del cliente.

In [3]:
print("Ruta del dataset original:")
print(RAW_PATH)

print("\n¿Existe el archivo CSV?")
print(RAW_PATH.exists())

Ruta del dataset original:
C:\Users\artur\OneDrive\Documents\Evolve\Python\Customer_Support_Ticket_Data_Project\data\raw\customer_support_tickets.csv

¿Existe el archivo CSV?
True


## 3. Carga del dataset original

Primero se carga el archivo CSV original. En esta etapa todavía no se aplica limpieza, para poder observar la estructura real del dataset recibido.

In [ ]:
df_raw = load_csv(RAW_PATH)

validate_raw_dataset(df_raw)

print(f"Número de filas: {df_raw.shape[0]}")
print(f"Número de columnas: {df_raw.shape[1]}")

df_raw.head()

## 4. Vista inicial del dataset

En esta sección se revisan las primeras filas, nombres de columnas, tipos de datos y una descripción estadística inicial.

Esto permite entender:

- qué columnas contiene el dataset,
- qué variables son numéricas o categóricas,
- qué columnas necesitan conversión de tipo,
- qué posibles problemas de calidad de datos pueden existir.

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all").T

In [ ]:
pd.DataFrame({
    "column_name": df_raw.columns,
    "dtype": df_raw.dtypes.astype(str),
    "non_null_count": df_raw.notna().sum().values,
    "null_count": df_raw.isna().sum().values,
    "null_percentage": (df_raw.isna().mean().values * 100).round(2)
})

## 5. Diccionario de columnas

A continuación se documentan las columnas principales del dataset y su interpretación dentro del análisis.

Durante la limpieza, los nombres originales se convertirán a formato `snake_case` para facilitar el trabajo con Python.

In [ ]:
column_dictionary = pd.DataFrame({
    "original_column": [
        "Ticket ID",
        "Customer Name",
        "Customer Email",
        "Customer Age",
        "Customer Gender",
        "Product Purchased",
        "Date of Purchase",
        "Ticket Type",
        "Ticket Subject",
        "Ticket Description",
        "Ticket Status",
        "Resolution",
        "Ticket Priority",
        "Ticket Channel",
        "First Response Time",
        "Time to Resolution",
        "Customer Satisfaction Rating",
    ],
    "clean_column": [
        "ticket_id",
        "customer_name",
        "customer_email",
        "customer_age",
        "customer_gender",
        "product_purchased",
        "date_of_purchase",
        "ticket_type",
        "ticket_subject",
        "ticket_description",
        "ticket_status",
        "resolution",
        "ticket_priority",
        "ticket_channel",
        "first_response_time",
        "time_to_resolution",
        "customer_satisfaction_rating",
    ],
    "description": [
        "Identificador único del ticket.",
        "Nombre del cliente. Se elimina por privacidad.",
        "Email del cliente. Se elimina por privacidad.",
        "Edad del cliente.",
        "Género del cliente.",
        "Producto asociado al ticket.",
        "Fecha de compra del producto.",
        "Categoría general del ticket.",
        "Tema específico del ticket.",
        "Descripción textual del problema. Se elimina para mantener el análisis estructurado.",
        "Estado actual del ticket.",
        "Texto de resolución. Se elimina por ser texto libre y no necesario para este EDA.",
        "Prioridad asignada al ticket.",
        "Canal por el cual se recibió el ticket.",
        "Fecha/hora de primera respuesta.",
        "Fecha/hora de resolución.",
        "Puntuación de satisfacción del cliente.",
    ]
})

column_dictionary

## 6. Calidad de datos inicial

Antes de limpiar el dataset, se revisan tres aspectos básicos:

1. Valores nulos.
2. Filas duplicadas.
3. Posibles duplicados por `Ticket ID`.

Esta revisión ayuda a decidir qué transformaciones aplicar y qué limitaciones mencionar en las conclusiones.

In [ ]:
missing_summary = (
    df_raw.isna()
    .sum()
    .reset_index()
)

missing_summary.columns = ["column", "missing_values"]
missing_summary["missing_percentage"] = (
    missing_summary["missing_values"] / len(df_raw) * 100
).round(2)

missing_summary.sort_values("missing_values", ascending=False)

In [ ]:
exact_duplicates = df_raw.duplicated().sum()
ticket_id_duplicates = df_raw["Ticket ID"].duplicated().sum()

print(f"Filas duplicadas exactas: {exact_duplicates}")
print(f"Ticket ID duplicados: {ticket_id_duplicates}")

## 7. Limpieza aplicada

La limpieza se realiza usando funciones creadas en `src/cleaning.py`.

Las principales decisiones de limpieza fueron:

- Convertir nombres de columnas a `snake_case`.
- Eliminar columnas con información personal o texto libre de alto ruido:
  - `customer_name`
  - `customer_email`
  - `ticket_description`
  - `resolution`
- Convertir columnas de fecha a formato datetime.
- Convertir variables numéricas al tipo adecuado.
- Normalizar categorías como prioridad, canal, estado y tipo de ticket.
- Eliminar duplicados exactos y duplicados por `ticket_id`.

Eliminar datos personales es una buena práctica porque permite enfocar el análisis en métricas operativas sin exponer información sensible.

In [ ]:
df_clean = clean(df_raw)

print(f"Filas antes de limpieza: {df_raw.shape[0]}")
print(f"Columnas antes de limpieza: {df_raw.shape[1]}")
print(f"Filas después de limpieza: {df_clean.shape[0]}")
print(f"Columnas después de limpieza: {df_clean.shape[1]}")

df_clean.head()

In [ ]:
removed_columns = sorted(set(df_raw.columns.str.lower().str.replace(" ", "_")) - set(df_clean.columns))
current_columns = list(df_clean.columns)

print("Columnas actuales después de limpieza:")
print(current_columns)

In [ ]:
df_clean.dtypes

## 8. Feature engineering

Después de limpiar el dataset, se crean nuevas variables para facilitar el análisis.

Features creadas:

- `product_age_days`: días entre la compra del producto y la primera respuesta.
- `resolution_hours`: horas entre la primera respuesta y la resolución.
- `has_negative_resolution_time`: identifica tickets donde la fecha de resolución aparece antes que la primera respuesta.
- `has_resolution`: indica si el ticket tiene fecha de resolución.
- `satisfaction_group`: agrupa la satisfacción en Low, Medium, High o No Rating.
- `is_high_priority`: identifica tickets High o Critical.
- `first_response_date`, `first_response_year`, `first_response_month`, `first_response_hour`: variables temporales derivadas de la primera respuesta.

Una decisión importante fue tratar los tiempos de resolución negativos como inconsistencias de calidad de datos. Estos registros se marcan con `has_negative_resolution_time` y su `resolution_hours` se convierte a valor nulo para evitar distorsionar el análisis.

In [ ]:
df = build_features(df_clean)

validate_clean_dataset(df)

print(f"Filas finales: {df.shape[0]}")
print(f"Columnas finales: {df.shape[1]}")

df.head()

In [ ]:
feature_columns = [
    "product_age_days",
    "resolution_hours",
    "has_negative_resolution_time",
    "has_resolution",
    "satisfaction_group",
    "is_high_priority",
    "first_response_date",
    "first_response_year",
    "first_response_month",
    "first_response_hour",
]

df[feature_columns].head(10)

In [ ]:
negative_resolution_cases = df["has_negative_resolution_time"].sum()
valid_resolution_cases = df["resolution_hours"].notna().sum()
tickets_with_resolution = df["has_resolution"].sum()

print(f"Tickets con inconsistencia temporal en resolución: {negative_resolution_cases}")
print(f"Tickets con tiempo de resolución válido: {valid_resolution_cases}")
print(f"Tickets con fecha de resolución registrada: {tickets_with_resolution}")

## 9. Análisis exploratorio

En esta sección se revisan distribuciones generales del dataset limpio:

- distribución de edades,
- distribución de satisfacción,
- volumen por tipo de ticket,
- volumen por prioridad,
- volumen por canal,
- estado de los tickets.

Esto permite entender el comportamiento general antes de analizar relaciones entre variables.

In [ ]:
numeric_columns = [
    "customer_age",
    "customer_satisfaction_rating",
    "product_age_days",
    "resolution_hours",
]

df[numeric_columns].describe().T

In [ ]:
categorical_columns = [
    "customer_gender",
    "ticket_type",
    "ticket_subject",
    "ticket_status",
    "ticket_priority",
    "ticket_channel",
    "satisfaction_group",
]

for column in categorical_columns:
    print(f"\n=== {column} ===")
    display(df[column].value_counts(dropna=False).to_frame("count"))

## 10. Visualización 1 — Tickets por tipo

Pregunta de análisis:

**¿Qué tipos de tickets son más frecuentes?**

Este gráfico permite identificar qué categorías generan mayor volumen de trabajo para el equipo de soporte.

In [ ]:
plot_tickets_by_type(df, save=False)
plt.show()

### Interpretación

Este gráfico muestra la distribución de tickets por tipo. Las categorías con mayor volumen representan las áreas donde el equipo de soporte recibe más demanda.

Si una categoría concentra muchos tickets, podría indicar:

- necesidad de mejorar documentación o autoservicio,
- problemas recurrentes en productos o procesos,
- oportunidad para crear guías, FAQs o automatizaciones.

## 11. Visualización 2 — Tickets por prioridad

Pregunta de análisis:

**¿Cómo se distribuyen los tickets según prioridad?**

Este gráfico ayuda a entender la carga operativa según criticidad.

In [ ]:
plot_tickets_by_priority(df, save=False)
plt.show()

### Interpretación

La distribución por prioridad permite evaluar si la operación está dominada por tickets de baja/media prioridad o si existe una alta proporción de tickets críticos.

Una proporción elevada de tickets `High` o `Critical` puede ser una señal de:

- problemas urgentes recurrentes,
- mala clasificación inicial,
- necesidad de revisar criterios de priorización,
- presión operativa sobre el equipo de soporte.

## 12. Visualización 3 — Tiempo de resolución por prioridad

Pregunta de análisis:

**¿Los tickets críticos tardan más o menos en resolverse?**

Para este gráfico se usa `resolution_hours`, excluyendo registros con tiempos negativos o faltantes.

In [ ]:
plot_resolution_time_by_priority(df, save=False)
plt.show()

### Interpretación

El boxplot permite comparar la distribución del tiempo de resolución entre prioridades.

Aspectos a observar:

- mediana de tiempo de resolución por prioridad,
- dispersión de los tiempos,
- presencia de outliers,
- diferencias entre tickets Low, Medium, High y Critical.

Si los tickets críticos no se resuelven más rápido que los de menor prioridad, podría ser necesario revisar el proceso de escalamiento o los acuerdos de nivel de servicio.

## 13. Visualización 4 — Satisfacción promedio por canal

Pregunta de análisis:

**¿Qué canal de soporte tiene mejor satisfacción promedio?**

Este gráfico compara la satisfacción media del cliente según el canal por el cual se gestionó el ticket.

In [ ]:
plot_satisfaction_by_channel(df, save=False)
plt.show()

### Interpretación

La satisfacción promedio por canal ayuda a identificar diferencias en la experiencia del cliente.

Si un canal tiene una satisfacción más baja, podría deberse a:

- tiempos de espera más altos,
- menor claridad en la comunicación,
- complejidad de los casos recibidos por ese canal,
- expectativas diferentes del usuario.

Este análisis puede apoyar decisiones sobre capacitación, documentación o redistribución de carga entre canales.

## 14. Visualización 5 — Tiempo de resolución vs satisfacción

Pregunta de análisis:

**¿Los tiempos de resolución más largos se asocian con menor satisfacción del cliente?**

Este gráfico explora la relación entre `resolution_hours` y `customer_satisfaction_rating`.

In [ ]:
plot_resolution_vs_satisfaction(df, save=False)
plt.show()

### Interpretación

Este gráfico permite observar si existe una tendencia entre el tiempo de resolución y la satisfacción.

Una relación negativa sugeriría que, a mayor tiempo de resolución, menor satisfacción. Sin embargo, si los puntos están muy dispersos, puede indicar que la satisfacción depende de otros factores además del tiempo, como la calidad de la comunicación, el tipo de problema, la prioridad o el canal de atención.

## 15. Visualización 6 — Productos con más tickets

Pregunta de análisis:

**¿Qué productos generan mayor carga de soporte?**

Este gráfico muestra los productos con mayor número de tickets registrados.

In [ ]:
plot_top_products_by_tickets(df, top_n=10, save=False)
plt.show()

### Interpretación

Los productos con mayor volumen de tickets pueden representar:

- productos más vendidos,
- productos más complejos,
- productos con problemas recurrentes,
- necesidad de mejorar documentación, configuración inicial o soporte preventivo.

Este análisis no necesariamente indica que un producto sea “peor”, ya que el volumen de tickets también puede depender del volumen de ventas. Para confirmarlo, sería ideal contar con datos de unidades vendidas por producto.

## 16. Análisis adicional — Matriz de prioridad vs estado

Además de los gráficos anteriores, se puede analizar cómo se distribuyen los estados de tickets según prioridad.

Esto ayuda a responder preguntas como:

- ¿Qué proporción de tickets críticos sigue abierta?
- ¿Hay muchas solicitudes pendientes de respuesta del cliente?
- ¿Los tickets de alta prioridad se cierran con mayor frecuencia?## 16. Análisis adicional — Matriz de prioridad vs estado

Además de los gráficos anteriores, se puede analizar cómo se distribuyen los estados de tickets según prioridad.

Esto ayuda a responder preguntas como:

- ¿Qué proporción de tickets críticos sigue abierta?
- ¿Hay muchas solicitudes pendientes de respuesta del cliente?
- ¿Los tickets de alta prioridad se cierran con mayor frecuencia?## 16. Análisis adicional — Matriz de prioridad vs estado

Además de los gráficos anteriores, se puede analizar cómo se distribuyen los estados de tickets según prioridad.

Esto ayuda a responder preguntas como:

- ¿Qué proporción de tickets críticos sigue abierta?
- ¿Hay muchas solicitudes pendientes de respuesta del cliente?
- ¿Los tickets de alta prioridad se cierran con mayor frecuencia?

In [ ]:
priority_status_table = pd.crosstab(
    df["ticket_priority"],
    df["ticket_status"],
    normalize="index"
).round(3) * 100

priority_status_table

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

sns.heatmap(
    priority_status_table,
    annot=True,
    fmt=".1f",
    cmap="Blues",
    ax=ax
)

ax.set_title("Distribución porcentual de estado del ticket por prioridad")
ax.set_xlabel("Estado del ticket")
ax.set_ylabel("Prioridad")

plt.tight_layout()
plt.show()

### Interpretación

La matriz permite comparar el estado de los tickets dentro de cada nivel de prioridad.

Un porcentaje alto de tickets críticos abiertos o pendientes podría indicar riesgo operativo, especialmente si esos tickets requieren atención rápida. También puede mostrar si una parte importante del backlog depende de respuesta del cliente.

## 17. Resumen de hallazgos calculados

En esta sección se calculan métricas resumen para apoyar las conclusiones finales con evidencia numérica.

In [ ]:
total_tickets = len(df)

top_ticket_type = df["ticket_type"].value_counts().idxmax()
top_ticket_type_count = df["ticket_type"].value_counts().max()

top_priority = df["ticket_priority"].value_counts().idxmax()
top_priority_count = df["ticket_priority"].value_counts().max()

top_channel = df["ticket_channel"].value_counts().idxmax()
top_channel_count = df["ticket_channel"].value_counts().max()

top_product = df["product_purchased"].value_counts().idxmax()
top_product_count = df["product_purchased"].value_counts().max()

avg_satisfaction = df["customer_satisfaction_rating"].mean()
median_resolution_hours = df["resolution_hours"].median()

negative_resolution_cases = df["has_negative_resolution_time"].sum()

summary_metrics = pd.DataFrame({
    "metric": [
        "Total tickets",
        "Most frequent ticket type",
        "Most frequent priority",
        "Most frequent channel",
        "Product with most tickets",
        "Average satisfaction rating",
        "Median valid resolution hours",
        "Negative resolution time cases",
    ],
    "value": [
        total_tickets,
        f"{top_ticket_type} ({top_ticket_type_count})",
        f"{top_priority} ({top_priority_count})",
        f"{top_channel} ({top_channel_count})",
        f"{top_product} ({top_product_count})",
        round(avg_satisfaction, 2),
        round(median_resolution_hours, 2),
        negative_resolution_cases,
    ]
})

summary_metrics

## 18. Hallazgos principales

A partir del análisis realizado, se identifican los siguientes hallazgos:

1. **La carga de soporte se concentra en ciertos tipos de tickets.**  
   El gráfico de tickets por tipo permite identificar qué categorías generan más demanda operativa.

2. **La prioridad de los tickets permite entender la presión del equipo de soporte.**  
   Si una proporción relevante de tickets es `High` o `Critical`, el equipo puede necesitar revisar criterios de priorización, escalamiento o distribución de carga.

3. **Existen inconsistencias temporales en algunos registros.**  
   Durante la creación de `resolution_hours`, se detectaron casos donde `time_to_resolution` era anterior a `first_response_time`. Estos registros fueron marcados con `has_negative_resolution_time` y excluidos del cálculo válido de tiempos de resolución.

4. **La satisfacción del cliente puede variar por canal de atención.**  
   Comparar satisfacción promedio por canal ayuda a identificar posibles diferencias en experiencia de usuario.

5. **Algunos productos generan mayor volumen de tickets.**  
   Esto puede indicar mayor base de usuarios, mayor complejidad del producto o necesidad de mejorar documentación y soporte preventivo.

## 19. Conclusiones

Este proyecto permitió construir un EDA reproducible sobre tickets de soporte técnico usando Python, Pandas, Matplotlib y Seaborn.

Las principales conclusiones son:

- El dataset permite analizar métricas operativas relevantes para un equipo de soporte, como volumen de tickets, prioridad, canal, resolución y satisfacción.
- La limpieza fue necesaria para estandarizar nombres de columnas, convertir fechas, normalizar categorías y eliminar información personal.
- La creación de features como `resolution_hours`, `product_age_days`, `satisfaction_group` e `is_high_priority` permitió enriquecer el análisis.
- Se detectaron inconsistencias temporales en algunos tickets, lo que demuestra la importancia de validar la calidad de los datos antes de generar conclusiones.
- Las visualizaciones ayudan a transformar datos operativos en información útil para la toma de decisiones.

## 20. Próximos pasos

Con más tiempo, este análisis podría ampliarse con:

- Un análisis temporal más profundo por día, semana o mes.
- Métricas tipo SLA, por ejemplo tickets resueltos dentro o fuera de un objetivo de tiempo.
- Segmentación por producto, canal y prioridad.
- Un dashboard en Power BI o Streamlit.
- Un modelo simple para predecir satisfacción o prioridad del ticket.
- Análisis de texto sobre `ticket_description`, si se quisiera explorar procesamiento de lenguaje natural.

Para una versión de portafolio, una mejora natural sería convertir este EDA en un dashboard de soporte con KPIs como:

- total de tickets,
- tickets abiertos,
- tickets críticos,
- tiempo medio de resolución,
- satisfacción promedio,
- productos con mayor carga de soporte.

In [ ]:
# Exportar dataset final procesado desde el notebook

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)

print(f"Dataset procesado guardado en: {OUT_PATH}")

In [ ]:
# Guardar visualizaciones finales en reports/figures/

FIGURES_PATH.mkdir(parents=True, exist_ok=True)

plot_tickets_by_type(df, save=True)
plot_tickets_by_priority(df, save=True)
plot_resolution_time_by_priority(df, save=True)
plot_satisfaction_by_channel(df, save=True)
plot_resolution_vs_satisfaction(df, save=True)
plot_top_products_by_tickets(df, top_n=10, save=True)

print(f"Visualizaciones guardadas en: {FIGURES_PATH}")

## 21. Cierre

Este notebook documenta el proceso completo de análisis exploratorio de datos aplicado a tickets de soporte técnico.

El resultado final es un pipeline reproducible que:

- carga el dataset original,
- valida su estructura,
- aplica limpieza,
- crea variables nuevas,
- genera visualizaciones,
- exporta un dataset procesado,
- y documenta hallazgos basados en evidencia.

Este enfoque conecta análisis de datos con un caso práctico de operaciones IT y soporte al cliente.